In [10]:
!pip install -q transformers sentencepiece language-tool-python sacrebleu rouge-score sentence-transformers


This cell installs the necessary Python libraries for this project. These libraries include:
- `transformers`: For working with pre-trained models like T5 for paraphrasing.
- `sentencepiece`: A dependency often used by `transformers` for tokenization.
- `language-tool-python`: For grammar and spell checking.
- `sacrebleu`: For calculating the BLEU score, a metric for evaluating machine translation and text generation.
- `rouge-score`: For calculating ROUGE scores, another set of metrics for evaluating text summarization and generation.
- `sentence-transformers`: For computing semantic similarity between sentences.

In [11]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import language_tool_python
from sentence_transformers import SentenceTransformer, util
from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer

import torch

# Load T5 model for paraphrasing. Using a publicly available generic T5 model.
tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

# Load grammar checker
grammar_tool = language_tool_python.LanguageTool('en-US')

# Load sentence-transformer for semantic similarity
sim_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Models and tools loaded successfully.")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Models and tools loaded successfully.


This cell loads the core models and tools required for paraphrasing, grammar correction, and semantic similarity:
- **T5 Tokenizer and Model (`t5-small`)**: Initializes a tokenizer and a conditional generation model from the `transformers` library. The `t5-small` model is a smaller version of the T5 (Text-to-Text Transfer Transformer) model, capable of various text-to-text tasks, including paraphrasing. The tokenizer is used to convert text into a format the model can understand, and vice versa.
- **LanguageTool (`grammar_tool`)**: Initializes `language_tool_python` for grammar and style checking. It uses 'en-US' for American English.
- **Sentence-Transformer (`all-MiniLM-L6-v2`)**: Loads a pre-trained `SentenceTransformer` model, specifically `all-MiniLM-L6-v2`. This model is used to convert sentences into numerical vector representations (embeddings), which can then be used to calculate semantic similarity.

In [12]:
def paraphrase_text(
    text,
    max_length=64,
    num_beams=5,
    no_repeat_ngram_size=2,
    num_return_sequences=1
):
    """
    Generate a paraphrase for the given input text using T5.
    """
    # Re-added 'paraphrase: ' prefix for generic T5 models
    input_text = "paraphrase: " + text

    input_ids = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding="longest"
    ).input_ids

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_length=max_length,
            num_beams=num_beams,
            no_repeat_ngram_size=no_repeat_ngram_size,
            early_stopping=True,
            num_return_sequences=num_return_sequences
        )

    paraphrases = [
        tokenizer.decode(o, skip_special_tokens=True)
        for o in outputs
    ]
    return paraphrases[0] if num_return_sequences == 1 else paraphrases

This cell defines the `paraphrase_text` function, which takes a given text and generates a paraphrase using the loaded T5 model. Here's a breakdown of its functionality:
- **`input_text = "paraphrase: " + text`**: Prepends "paraphrase: " to the input text. This is a common practice when using generic T5 models, as it helps guide the model to perform a paraphrasing task.
- **`tokenizer(input_text, ...)`**: Encodes the input text into numerical `input_ids` that the T5 model can process. `return_tensors="pt"` ensures PyTorch tensors are returned, `truncation=True` handles texts longer than the model's maximum input length, and `padding="longest"` pads shorter texts.
- **`model.generate(...)`**: Generates new text based on the `input_ids`. Key parameters include:
    - `max_length`: Sets the maximum length of the generated paraphrase.
    - `num_beams`: Uses beam search with 5 beams to find more diverse and high-quality paraphrases.
    - `no_repeat_ngram_size`: Prevents the generation of repetitive n-grams (sequences of words), improving fluency.
    - `early_stopping=True`: Stops generation early if a complete sequence is found.
    - `num_return_sequences`: Specifies how many paraphrases to generate (default is 1).
- **`tokenizer.decode(o, skip_special_tokens=True)`**: Decodes the generated `output_ids` back into human-readable text, skipping any special tokens used by the model.

In [13]:
def correct_grammar(text: str) -> str:
    """
    Use LanguageTool to correct grammar, spelling, and basic fluency issues.
    """
    return grammar_tool.correct(text)


def paraphrase_with_correction(text: str) -> str:
    """
    Full pipeline: paraphrase + grammar/fluency correction.
    """
    raw_paraphrase = paraphrase_text(text)
    corrected = correct_grammar(raw_paraphrase)
    return corrected


This cell defines two functions:
- **`correct_grammar(text: str) -> str`**: This function uses the `grammar_tool` (LanguageTool) initialized earlier to identify and correct grammar, spelling, and basic fluency issues in the input text. It takes a string as input and returns the corrected string.
- **`paraphrase_with_correction(text: str) -> str`**: This function combines the paraphrasing and grammar correction steps. It first calls `paraphrase_text` to generate a paraphrase and then passes this raw paraphrase to `correct_grammar` for refinement. This creates a pipeline for generating grammatically sound paraphrases.

In [14]:
def semantic_similarity(text1: str, text2: str) -> float:
    """
    Compute cosine similarity between two sentences using Sentence-BERT.
    """
    emb1 = sim_model.encode(text1, convert_to_tensor=True)
    emb2 = sim_model.encode(text2, convert_to_tensor=True)
    sim = util.pytorch_cos_sim(emb1, emb2).item()
    return float(sim)


This cell defines the `semantic_similarity` function, which quantifies how similar in meaning two given sentences are:
- **`emb1 = sim_model.encode(text1, convert_to_tensor=True)`**: Converts the first input text (`text1`) into a numerical vector embedding using the `sim_model` (Sentence-Transformer). `convert_to_tensor=True` ensures the output is a PyTorch tensor.
- **`emb2 = sim_model.encode(text2, convert_to_tensor=True)`**: Similarly, converts the second input text (`text2`) into its embedding.
- **`sim = util.pytorch_cos_sim(emb1, emb2).item()`**: Computes the cosine similarity between the two sentence embeddings. Cosine similarity measures the cosine of the angle between two vectors; a value closer to 1 indicates higher semantic similarity, while a value closer to 0 indicates lower similarity. `.item()` extracts the scalar value from the tensor.

In [15]:
def compute_bleu(references, hypotheses):
    """
    Compute corpus BLEU score.
    references: list of reference strings
    hypotheses: list of hypothesis strings
    """
    # sacrebleu expects list of list for references
    bleu = corpus_bleu(hypotheses, [references])
    return bleu.score


def compute_rouge(references, hypotheses):
    """
    Compute average ROUGE-1, ROUGE-2, ROUGE-L F1 scores.
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    r1, r2, rl = 0.0, 0.0, 0.0

    for ref, hyp in zip(references, hypotheses):
        scores = scorer.score(ref, hyp)
        r1 += scores['rouge1'].fmeasure
        r2 += scores['rouge2'].fmeasure
        rl += scores['rougeL'].fmeasure

    n = len(hypotheses)
    return {
        "rouge1": r1 / n,
        "rouge2": r2 / n,
        "rougeL": rl / n
    }


This cell defines functions for evaluating the quality of the generated paraphrases using established metrics:
- **`compute_bleu(references, hypotheses)`**: This function calculates the BLEU (Bilingual Evaluation Understudy) score. BLEU is a metric for evaluating the quality of text which has been machine-translated or machine-generated from one natural language to another. It compares candidate translations to a set of high-quality reference translations. A higher BLEU score indicates a closer match to the reference.
- **`compute_rouge(references, hypotheses)`**: This function calculates ROUGE (Recall-Oriented Understudy for Gisting Evaluation) scores, specifically ROUGE-1, ROUGE-2, and ROUGE-L F1 scores. ROUGE is commonly used for evaluating automatic summarization and machine translation. It measures the overlap of n-grams (sequences of words) between the generated text (hypotheses) and reference texts. Higher ROUGE scores indicate more overlap and thus better quality.

In [19]:
sample_inputs = [
    "Data scientists analyze large datasets to extract meaningful insights and support business strategies.",
    "Cloud computing allows companies to scale their infrastructure without investing heavily in physical hardware.",
]

paraphrased_outputs = []
corrected_outputs = []
similarities = []

for text in sample_inputs:
    paraphrased = paraphrase_text(text)
    corrected = correct_grammar(paraphrased)
    sim = semantic_similarity(text, corrected)

    paraphrased_outputs.append(paraphrased)
    corrected_outputs.append(corrected)
    similarities.append(sim)

for i, (src, para, corr, sim) in enumerate(zip(sample_inputs, paraphrased_outputs, corrected_outputs, similarities)):
    print(f"\nExample {i+1}")
    print("Original:   ", src)
    print("Paraphrase: ", para)
    print("Corrected:  ", corr)
    print("Similarity: ", round(sim, 3))



Example 1
Original:    Data scientists analyze large datasets to extract meaningful insights and support business strategies.
Paraphrase:  Paraphrase: Data scientists analyze large datasets to extract meaningful insights and support business strategies.
Corrected:   Paraphrase: Data scientists analyze large datasets to extract meaningful insights and support business strategies.
Similarity:  0.878

Example 2
Original:    Cloud computing allows companies to scale their infrastructure without investing heavily in physical hardware.
Paraphrase:  Paraphrase: Cloud computing lets companies scale their infrastructure without investing heavily in physical hardware.
Corrected:   Paraphrase: Cloud computing lets companies scale their infrastructure without investing heavily in physical hardware.
Similarity:  0.95


This cell demonstrates the paraphrasing and grammar correction pipeline with a set of sample input sentences and then displays the results:
- **`sample_inputs`**: A list of example sentences related to technology and AI.
- **Loop for processing**: Iterates through each `text` in `sample_inputs`.
    - **`paraphrased = paraphrase_text(text)`**: Generates a raw paraphrase.
    - **`corrected = correct_grammar(paraphrased)`**: Corrects the grammar of the raw paraphrase.
    - **`sim = semantic_similarity(text, corrected)`**: Calculates the semantic similarity between the original sentence and the corrected paraphrase.
- **Storing results**: The generated paraphrases, corrected texts, and similarity scores are stored in respective lists.
- **Printing results**: Finally, it iterates through the results and prints the original sentence, its paraphrase, the grammar-corrected version, and the semantic similarity score for each example, providing a clear comparison.

In [20]:
# For metrics, we treat original sentences as references
references = sample_inputs
hypotheses = corrected_outputs  # final paraphrased + corrected

bleu_score = compute_bleu(references, hypotheses)
rouge_scores = compute_rouge(references, hypotheses)

avg_similarity = sum(similarities) / len(similarities)

print("\n=== Evaluation Report ===")
print(f"BLEU score:        {bleu_score:.2f}")
print(f"ROUGE-1 F1:        {rouge_scores['rouge1']:.3f}")
print(f"ROUGE-2 F1:        {rouge_scores['rouge2']:.3f}")
print(f"ROUGE-L F1:        {rouge_scores['rougeL']:.3f}")
print(f"Avg semantic sim.: {avg_similarity:.3f}")

print("\nInterpretation:")
print("- Higher BLEU and ROUGE indicate closer overlap with original meaning.")
print("- High semantic similarity (>0.8) suggests meaning is preserved.")
print("- Manual inspection of examples shows improved clarity and originality.")



=== Evaluation Report ===
BLEU score:        75.21
ROUGE-1 F1:        0.910
ROUGE-2 F1:        0.826
ROUGE-L F1:        0.910
Avg semantic sim.: 0.914

Interpretation:
- Higher BLEU and ROUGE indicate closer overlap with original meaning.
- High semantic similarity (>0.8) suggests meaning is preserved.
- Manual inspection of examples shows improved clarity and originality.


This cell computes and displays an evaluation report using the metrics defined earlier, along with an interpretation of the results:
- **`references = sample_inputs`**: The original sentences are used as references for evaluation.
- **`hypotheses = corrected_outputs`**: The grammar-corrected paraphrases are used as the generated text (hypotheses).
- **`bleu_score = compute_bleu(references, hypotheses)`**: Calculates the BLEU score.
- **`rouge_scores = compute_rouge(references, hypotheses)`**: Calculates ROUGE-1, ROUGE-2, and ROUGE-L F1 scores.
- **`avg_similarity = sum(similarities) / len(similarities)`**: Computes the average semantic similarity across all samples.
- **Printing the report**: Displays all the calculated metrics (BLEU, ROUGE scores, and average semantic similarity).
- **Interpretation**: Provides guidance on how to interpret the scores, noting that higher BLEU and ROUGE scores indicate closer overlap with the original meaning, and high semantic similarity suggests meaning preservation.

In [18]:
def interactive_paraphraser():
    while True:
        text = input("\nEnter a sentence to paraphrase (or 'quit'): ")
        if text.lower().strip() in ["quit", "exit"]:
            break
        para = paraphrase_with_correction(text)
        sim = semantic_similarity(text, para)
        print("Paraphrased:", para)
        print("Similarity:", round(sim, 3))

# Uncomment to use in Colab:
# interactive_paraphraser()


This cell defines an interactive function `interactive_paraphraser()` that allows users to input sentences and get immediate paraphrased and grammar-corrected outputs:
- **`while True:`**: Enters an infinite loop to allow continuous interaction.
- **`text = input(...)`**: Prompts the user to enter a sentence to paraphrase or 'quit' to exit.
- **`if text.lower().strip() in ["quit", "exit"]:`**: Checks if the user wants to quit the interactive session.
- **`para = paraphrase_with_correction(text)`**: Calls the combined paraphrasing and grammar correction function.
- **`sim = semantic_similarity(text, para)`**: Calculates the semantic similarity between the user's input and the generated paraphrase.
- **Printing results**: Displays the paraphrased text and its semantic similarity to the original input.
- **`# interactive_paraphraser()`**: The function call is commented out by default. To use the interactive mode, you would uncomment this line and run the cell.